# Lecture 4.1 — Understanding `allowed_tools` and Permission Modes

**Claude Agent SDK: Build AI Agents in Python — Section 4: Permissions & Safety**

This notebook is **independent and self-contained**.

In the lecture we learned the **6-gate permission evaluation flowchart**:

```
Gate 1  PreToolUse hooks
Gate 2  disallowed_tools   (deny rules)
Gate 3  allowed_tools      (allow rules)
Gate 4  permission mode
Gate 5  canUseTool callback
Gate 6  fallback (no callback registered)
```

Every demo is a controlled experiment that validates a specific gate claim:

| # | Claim | Gate |
|---|-------|------|
| 1 | `allowed_tools` does **not** block unlisted tools — they fall through | Gate 3 → Gate 4/6 |
| 2 | `disallowed_tools` beats `allowed_tools` — deny wins | Gate 2 before Gate 3 |
| 3 | `bypassPermissions` ignores `allowed_tools` but obeys `disallowed_tools` | Gate 3 vs Gate 4 vs Gate 2 |
| 4 | `dontAsk` denies unlisted tools outright — `canUseTool` never runs | Gate 5 / Gate 6 |

> **Important SDK behaviour to know before the demos:**
> Read-only tools — `Read`, `Glob`, `Grep` — are **auto-approved by the Claude Code CLI
> itself** regardless of `allowed_tools` or permission mode. The CLI classifies them as
> non-destructive and never routes them through the permission prompt flow at all.
> This means you cannot use them to test "will an unlisted tool be blocked?" — they
> will always succeed. **All demos that need a tool to be blockable use `Write` or
> `Bash`**, which do go through the full gate flow.

In [1]:
# Install the Claude Agent SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 9.2 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

Set `MODEL_NAME` to any Claude model you have access to.
https://platform.claude.com/docs/en/about-claude/models/overview

In [3]:
# Model configuration
# Change this to use a different Claude model
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview

MODEL_NAME = "claude-haiku-4-5"

## Creating Sample Files

In [4]:
import os

os.makedirs("/content/permissions_demo/src", exist_ok=True)

with open("/content/permissions_demo/src/auth.py", "w") as f:
    f.write("# Authentication module\n\n")
    f.write("def login(user, password):\n")
    f.write("    return user == 'admin'\n\n")
    f.write("def logout(user):\n")
    f.write("    return True\n")

with open("/content/permissions_demo/notes.txt", "w") as f:
    f.write("Project Notes\n=============\n")
    f.write("Auth module needs password hashing\n")
    f.write("Utils module needs input validation\n")

print("Files created.")

Files created.


## Colab Setup: Always Set `cwd="/content"`

Each built-in tool (Glob, Read, Write, Bash …) has its own internal path-scope check
that runs before the gate logic. If `cwd` is not set, the SDK defaults to a directory
outside `/content`, so tool calls targeting `/content/permissions_demo/…` are blocked
by the tool's own check before the gates we are studying even run.

Always pass `cwd="/content"` in every `ClaudeAgentOptions` call in Colab.

> **Real production gotcha:** if a tool is failing even though it is in `allowed_tools`,
> always check `cwd` first. The tool's own path-scope check fires before Gate 3.

## The `run_agent()` Helper

The helper collects every `ToolUseBlock.name` from the message stream — one per tool
call the model decided to make. The fact that a `ToolUseBlock` was emitted proves the
tool was **offered to and attempted by the model**.

We pair this with **filesystem ground truth** (did a file get created or survive?) to
prove each gate claim. We do not try to parse error messages from the stream.

In [5]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions,
    AssistantMessage, ResultMessage, ToolUseBlock,
)


async def run_agent(prompt, options, label=""):
    """Run an agent and print which tools it attempted.

    Collects ToolUseBlock names from AssistantMessages — one per tool call
    the model decided to make. Returns attempted list and final result.
    """
    attempted = []
    final_result = None

    if label:
        print("=" * 64)
        print(f"  {label}")
        print("=" * 64)

    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, ToolUseBlock):
                    attempted.append(block.name)
        if isinstance(message, ResultMessage):
            final_result = message.result

    unique = sorted(set(attempted))
    print(f"Tools ATTEMPTED : {unique if unique else '(none)'}")
    print(f"Final result    :\n{final_result}\n")
    return {"attempted": unique, "result": final_result}

## Part 1 — Claim 1: `allowed_tools` does NOT block unlisted tools (Gate 3 → Gate 4/6)

**The misconception:** `allowed_tools` is a whitelist that blocks everything not listed.
It is not — it is a *pre-approval list* at Gate 3. An unlisted tool still **falls
through** to Gates 4/5/6.

**Why we use `Write` here (not `Glob` or `Read`):**
Read-only tools are auto-approved by the CLI regardless of `allowed_tools` or mode.
`Write` goes through the full gate flow, so it is the right tool to test with.

**The experiment:** same task, same `dontAsk` mode, only `allowed_tools` changes.

- **Run A** — `allowed_tools=["Read", "Glob"]`, `Write` is **unlisted**.
  `dontAsk` denies the fall-through at Gate 6. `summary_a.txt` must **not exist**.

- **Run B** — `allowed_tools=["Read", "Glob", "Write"]`, `Write` is **listed**.
  Gate 3 approves it immediately. `summary_b.txt` **must exist**.

Same mode. Same model. Only `allowed_tools` changed. That contrast is the proof:
`allowed_tools` pre-approves listed tools but does **not** block or strip unlisted ones.
What stopped `Write` in Run A was **Gate 6**, not the list.

In [6]:
# Ensure no leftover output files from a previous run
output_a = "/content/permissions_demo/summary_a.txt"
output_b = "/content/permissions_demo/summary_b.txt"
for f in [output_a, output_b]:
    if os.path.exists(f): os.remove(f)

PROMPT_WRITE_A = """
Read the file permissions_demo/src/auth.py, then use the Write tool to save
a one-sentence summary of what it does to permissions_demo/summary_a.txt.
"""

# Run A — Write is NOT in allowed_tools. dontAsk denies the fall-through at Gate 6.
result_a = await run_agent(
    PROMPT_WRITE_A,
    ClaudeAgentOptions(
        cwd="/content",
        allowed_tools=["Read", "Glob"],   # Write deliberately omitted
        permission_mode="dontAsk",
        model=MODEL_NAME,
    ),
    label="RUN A — Write unlisted, dontAsk denies the fall-through",
)

  RUN A — Write unlisted, dontAsk denies the fall-through
Tools ATTEMPTED : ['Read', 'Write']
Final result    :
I've read the auth.py file successfully. However, I don't have permission to use the Write tool in the current mode. 

The file contains an authentication module with two simple functions:
- `login(user, password)`: returns True only if the user is 'admin'
- `logout(user)`: always returns True

To complete this task, I would need write permission enabled. Would you like me to request permission or use an alternative approach?



In [7]:
PROMPT_WRITE_B = """
Read the file permissions_demo/src/auth.py, then use the Write tool to save
a one-sentence summary of what it does to permissions_demo/summary_b.txt.
"""

# Run B — Write IS in allowed_tools. Gate 3 approves it immediately.
result_b = await run_agent(
    PROMPT_WRITE_B,
    ClaudeAgentOptions(
        cwd="/content",
        allowed_tools=["Read", "Glob", "Write"],  # Write now listed
        permission_mode="dontAsk",
        model=MODEL_NAME,
    ),
    label="RUN B — Write listed, Gate 3 approves it",
)

  RUN B — Write listed, Gate 3 approves it
Tools ATTEMPTED : ['Read', 'Write']
Final result    :
Done! I've read the `permissions_demo/src/auth.py` file and created a one-sentence summary in `permissions_demo/summary_b.txt` that describes what the authentication module does.



In [8]:
# ── VALIDATE CLAIM 1 ──────────────────────────────────────────────────────────
a_created = os.path.exists(output_a)
b_created = os.path.exists(output_b)

print(f"Run A (Write unlisted) — summary_a.txt created? {a_created}  (expected: False)")
print(f"Run B (Write listed)   — summary_b.txt created? {b_created}  (expected: True)")
print()

assert not a_created, "❌ CLAIM 1 FAILED: Write ran even though it was unlisted in dontAsk."
assert b_created,     "❌ CLAIM 1 FAILED: Write didn't run even though it was listed."
print("✅ CLAIM 1 CONFIRMED: allowed_tools is a pre-approval list, not a whitelist.")
print("   Run A: Write unlisted → fell through to Gate 6 → denied → no file.")
print("   Run B: Write listed   → approved at Gate 3    → ran   → file created.")
print("   The tool was never stripped. The GATE stopped it, not the list.")

Run A (Write unlisted) — summary_a.txt created? False  (expected: False)
Run B (Write listed)   — summary_b.txt created? True  (expected: True)

✅ CLAIM 1 CONFIRMED: allowed_tools is a pre-approval list, not a whitelist.
   Run A: Write unlisted → fell through to Gate 6 → denied → no file.
   Run B: Write listed   → approved at Gate 3    → ran   → file created.
   The tool was never stripped. The GATE stopped it, not the list.


## Part 2 — Claim 2: `disallowed_tools` beats `allowed_tools` (Gate 2 before Gate 3)

A deny rule at Gate 2 is checked *before* `allowed_tools` at Gate 3. If the same tool
is in both lists, the **deny wins**.

**Subtlety:** a *bare-name* deny like `disallowed_tools=["Bash"]` removes Bash from
context entirely — the model never sees it. To observe Gate 2 matching a *live call*
while Bash is still visible to the model, we use a **scoped deny**:
`disallowed_tools=["Bash(rm *)"]`. This keeps Bash available but denies any call
matching `rm *` — in every mode.

**Ground-truth proof:** allow Bash (Gate 3 would approve it), dare the agent to `rm`
a file. If Gate 2 wins, the **file still exists** afterward.

In [9]:
with open("/content/permissions_demo/delete_me.txt", "w") as f:
    f.write("If this file survives, the deny rule at Gate 2 won.\n")

print("Before:", os.path.exists("/content/permissions_demo/delete_me.txt"))

Before: True


In [10]:
PROMPT_DELETE = """
Use the Bash tool to delete the file permissions_demo/delete_me.txt
by running exactly: rm permissions_demo/delete_me.txt
Then confirm whether the deletion succeeded.
"""

# Bash is in allowed_tools → Gate 3 would approve it.
# The scoped deny at Gate 2 fires FIRST → blocks the rm call.
result_deny = await run_agent(
    PROMPT_DELETE,
    ClaudeAgentOptions(
        cwd="/content",
        allowed_tools=["Bash", "Glob", "Read"],
        disallowed_tools=["Bash(rm *)"],   # Gate 2 — scoped, fires before Gate 3
        permission_mode="default",
        model=MODEL_NAME,
    ),
    label="DENY beats ALLOW — Bash in allowed_tools, Bash(rm *) denied at Gate 2",
)

  DENY beats ALLOW — Bash in allowed_tools, Bash(rm *) denied at Gate 2
Tools ATTEMPTED : ['Bash']
Final result    :
The deletion was **denied** due to permissions restrictions. The Bash tool blocked the `rm` command on `permissions_demo/delete_me.txt` because this project has security policies in place that prevent deletion of files without explicit user authorization.

This is a protection mechanism to prevent accidental or unintended file deletions. If you need to delete this file, you would need to either:
1. Grant explicit permission for file deletion operations, or
2. Use a different mechanism if one is available in your project's configuration

Would you like to grant permission for this deletion or explore alternative approaches?



In [11]:
# ── VALIDATE CLAIM 2 ──────────────────────────────────────────────────────────
file_survived = os.path.exists("/content/permissions_demo/delete_me.txt")

print(f"Bash was ATTEMPTED     : {'Bash' in result_deny['attempted']}")
print(f"delete_me.txt survived : {file_survived}  (expected: True)")
print()

assert file_survived, "❌ CLAIM 2 FAILED: file was deleted — deny rule did not win."
print("✅ CLAIM 2 CONFIRMED: disallowed_tools beat allowed_tools.")
print("   Bash was in allowed_tools (Gate 3 would approve), but the scoped deny")
print("   at Gate 2 fired FIRST. The rm was blocked. The file survived.")

Bash was ATTEMPTED     : True
delete_me.txt survived : True  (expected: True)

✅ CLAIM 2 CONFIRMED: disallowed_tools beat allowed_tools.
   Bash was in allowed_tools (Gate 3 would approve), but the scoped deny
   at Gate 2 fired FIRST. The rm was blocked. The file survived.


## Part 3 — Claim 3: `bypassPermissions` ignores `allowed_tools` but obeys `disallowed_tools`

This is the **safety-critical gotcha** from the lecture.

> ⚠️ **`bypassPermissions` cannot run in Colab.** The Claude Code CLI refuses to run
> `--dangerously-skip-permissions` as root, and Colab runs as root →
> `ProcessError: exit code 1`. This part is therefore **explanatory only** — read the
> code block and reasoning below, but do not execute the cell.

---

**The gotcha explained:**

`allowed_tools` is Gate 3. `bypassPermissions` is Gate 4. These are **separate steps**.

When an unlisted tool falls through Gate 3, it lands at Gate 4 where `bypassPermissions`
approves it. The `allowed_tools` list never had a chance to stop it — it only pre-approves
at Gate 3, it does not block the fall-through.

So setting `allowed_tools=["Read"]` alongside `permission_mode="bypassPermissions"` still
approves `Bash`, `Write`, `Edit`, and everything else. **The list is irrelevant.**

**The fix:** use `disallowed_tools` at Gate 2 — the only gate that fires *before* the
mode check and holds in every mode including `bypassPermissions`.

```python
# NON-ROOT MACHINE ONLY — do NOT run in Colab (exits with ProcessError: exit code 1)
await run_agent(
    "Read auth.py and write a summary to permissions_demo/summary_bypass.txt",
    ClaudeAgentOptions(
        cwd="/content",
        allowed_tools=["Read"],                # only Read listed — Write is NOT here
        permission_mode="bypassPermissions",   # approves EVERYTHING at Gate 4
        model=MODEL_NAME,
    ),
)
# Expected on a non-root machine:
#   Tools ATTEMPTED : ['Read', 'Write']
#   summary_bypass.txt IS created — even though Write was not in allowed_tools.
#   Gate 4 (bypass) approved the fall-through. The list did nothing.
```

**To block a tool under `bypassPermissions`:**

```python
# This works — disallowed_tools at Gate 2 fires before bypass at Gate 4
ClaudeAgentOptions(
    disallowed_tools=["Bash(rm *)"],       # Gate 2 — holds in every mode
    permission_mode="bypassPermissions",
)
```

> Claim 2 (Part 2) already demonstrated that Gate 2 holds regardless of mode —
> the scoped deny blocked Bash even with it in `allowed_tools`. The same Gate 2 ordering
> applies to `bypassPermissions`. No separate runnable proof is needed.

## Part 4 — Claim 4: `dontAsk` denies unlisted tools outright (Gate 5 / Gate 6)

In `dontAsk` mode, anything not pre-approved by `allowed_tools` is **denied** — and the
`canUseTool` callback is **never called** (Gate 5 skipped, Gate 6 denies).

Part 1 Run A already showed this for `Write` in isolation. Here we make the guarantee
explicit for **multiple unlisted tools at once** — `Write` and `Bash` — and confirm
neither can run regardless of what the model tries.

**The experiment:** allow only `Read` and `Glob` in `dontAsk`, then ask the agent to
both write a file and run a shell command. Neither output should exist afterward.

In [12]:
# Clean up any artifacts from previous runs
for f in ["/content/permissions_demo/dontask_output.txt", "/tmp/bash_test.txt"]:
    if os.path.exists(f): os.remove(f)

PROMPT_MULTI = """
Please do two things:
1. Use the Write tool to create permissions_demo/dontask_output.txt with the text "created"
2. Use the Bash tool to run: echo "hello" > /tmp/bash_test.txt
Then tell me what succeeded.
"""

# Only Read and Glob are allowed. Write and Bash are unlisted.
# dontAsk denies both at Gate 6 — no canUseTool, no escape.
result_dontask = await run_agent(
    PROMPT_MULTI,
    ClaudeAgentOptions(
        cwd="/content",
        allowed_tools=["Read", "Glob"],   # Write and Bash NOT listed
        permission_mode="dontAsk",
        model=MODEL_NAME,
    ),
    label="dontAsk — only Read/Glob allowed, Write and Bash denied at Gate 6",
)

  dontAsk — only Read/Glob allowed, Write and Bash denied at Gate 6
Tools ATTEMPTED : ['Bash', 'Write']
Final result    :
Both operations were **denied** due to Claude Code running in "don't ask mode". 

Here's what happened:

1. **Write tool** - Permission denied to create `permissions_demo/dontask_output.txt`
2. **Bash tool** - Permission denied to run the echo command

This is a deliberate security feature. When Claude Code is in "don't ask mode," certain tools that can modify the filesystem or run arbitrary commands are disabled to prevent unintended side effects. This protects against accidental or malicious modifications.

To proceed, you would need to either:
- Exit "don't ask mode" and grant explicit permission for these tools
- Use the `/config` or `update-config` skill to adjust the permission settings
- Ask me to request permission (which would show you a permission dialog)

Would you like me to attempt to get permission to use these tools?



In [13]:
# ── VALIDATE CLAIM 4 ──────────────────────────────────────────────────────────
write_blocked = not os.path.exists("/content/permissions_demo/dontask_output.txt")
bash_blocked  = not os.path.exists("/tmp/bash_test.txt")

print(f"dontask_output.txt NOT created : {write_blocked}  (expected: True — Write denied)")
print(f"bash_test.txt NOT created      : {bash_blocked}   (expected: True — Bash denied)")
print(f"Tools ATTEMPTED                : {result_dontask['attempted']}")
print()

assert write_blocked, "❌ CLAIM 4 FAILED: Write ran in dontAsk mode."
assert bash_blocked,  "❌ CLAIM 4 FAILED: Bash ran in dontAsk mode."
print("✅ CLAIM 4 CONFIRMED: dontAsk denied all unlisted tools outright.")
print("   Write and Bash could not run. Gate 5 skipped (no canUseTool).")
print("   Gate 6 denied. No callback could have saved them.")

dontask_output.txt NOT created : True  (expected: True — Write denied)
bash_test.txt NOT created      : True   (expected: True — Bash denied)
Tools ATTEMPTED                : ['Bash', 'Write']

✅ CLAIM 4 CONFIRMED: dontAsk denied all unlisted tools outright.
   Write and Bash could not run. Gate 5 skipped (no canUseTool).
   Gate 6 denied. No callback could have saved them.


## Summary — What We Proved

| Claim | Evidence |
|-------|----------|
| **1. `allowed_tools` is not a whitelist** | `summary_a.txt` not created (Write unlisted, `dontAsk`); `summary_b.txt` created (Write listed, same mode). Same mode — only the list changed. |
| **2. Deny beats allow** | `delete_me.txt` survived despite Bash being in `allowed_tools`. Scoped deny at Gate 2 fired first. |
| **3. `bypassPermissions` gotcha** | Explanatory: `allowed_tools` is irrelevant under bypass because Gate 3 and Gate 4 are separate. Unlisted tools fall through and are approved at Gate 4. Fix: `disallowed_tools` at Gate 2. |
| **4. `dontAsk` denies the fall-through** | `dontask_output.txt` and `bash_test.txt` not created. Both unlisted tools denied at Gate 6. No escape. |

**Key insight from building these demos:**
Read-only tools (`Read`, `Glob`, `Grep`) are auto-approved by the CLI itself —
they never enter the gate flow. To test gate behaviour, use write-capable tools.

**The two big lessons:**

1. **`allowed_tools` pre-approves; it never strips.** What stops an unlisted
   write-capable tool is the mode at Gates 4/5/6 — not the list.

2. **`disallowed_tools` is the only absolute block.** A scoped deny at Gate 2 holds
   in every mode including `bypassPermissions`.

**The locked-down pattern:** `allowed_tools` + `permission_mode="dontAsk"` — listed
tools approved at Gate 3, everything else denied at Gate 6, no callback, no escape.

---
**Coming up in Lecture 4.2 — Read-Only Agents vs Auto-Edit Agents.**